<a href="https://colab.research.google.com/github/MohamedMohsenKoresh/Car_Prediction/blob/main/CV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers joblib scikit-learn pandas openpyxl

from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix
import joblib
import os
import re

# Load data file (ensure column names are 'Resume' and 'Category')
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

if "Resume" not in df.columns or "Category" not in df.columns:
    raise ValueError("The CSV must contain 'Resume' and 'Category' columns")

# Improved text cleaning function to remove links and formatting symbols
def clean_text(text):
    text = str(text).lower()
    # Remove links and email
    text = re.sub(r'http\S+|www\S+|@\S+|[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', ' ', text)
    # Remove common formatting symbols in resumes (list bullets)
    text = re.sub(r'[\u2022\u2023\u25e6\u2043\u203B]', ' ', text)
    # Remove symbols except letters, numbers, and spaces
    text = re.sub(r'[^a-zA-Z0-9أ-ي\s]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Data cleaning and preparation
texts = [clean_text(t) for t in df["Resume"].astype(str).tolist()]
labels = df["Category"].astype(str).tolist()

# TF-IDF Features - increasing features to 5000
print("Generating TF-IDF embeddings...")
MAX_FEATURES = 5000

word_vec = TfidfVectorizer(max_features=MAX_FEATURES, analyzer='word')
x_w = word_vec.fit_transform(texts)

char_vec = TfidfVectorizer(max_features=MAX_FEATURES, analyzer='char', ngram_range=(2, 4))
x_c = char_vec.fit_transform(texts)

# BERT Embeddings
EMBEDDER_DIR = "embedder"
print("Loading BERT model...")

# all-MiniLM-L6-v2 model
bert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Generating BERT embeddings...")
bert_embeddings = bert_model.encode(texts, show_progress_bar=True)
emb_sparse = csr_matrix(bert_embeddings)

# Combine TF-IDF + BERT
print("Combining TF-IDF + BERT...")
X_hybrid = hstack([x_w, x_c, emb_sparse]).tocsr()

# Split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X_hybrid, labels, test_size=0.2, random_state=42
)

# Train LinearSVC using Class Weights to improve rare class accuracy
print("Training LinearSVC model with class weights...")
model = LinearSVC(max_iter=5000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")

# Detailed Performance Evaluation (Classification Report)
print("\n" + "="*50)
print("Classification Report:")

y_pred = model.predict(X_test)

le = LabelEncoder()
y_test_encoded = le.fit_transform(y_test)
y_pred_encoded = le.transform(y_pred)

print(classification_report(y_test_encoded, y_pred_encoded, target_names=le.classes_, zero_division=0))
print("="*50 + "\n")


# Save Models and Vectorizers for use in app.py
vecs_to_save = {
    "word": word_vec,
    "char": char_vec
}
joblib.dump(vecs_to_save, "tfidf_vectorizer.joblib")
joblib.dump(model, "hybrid_model.joblib")

# Save BERT Model to folder
if not os.path.exists(EMBEDDER_DIR):
    bert_model.save(EMBEDDER_DIR)
    print(f"BERT Embedder saved to directory: {EMBEDDER_DIR}")

# Download files from Google Colab
print("\nStarting download of model files...")
try:
    files.download("hybrid_model.joblib")
    files.download("tfidf_vectorizer.joblib")
    # Zip folder first
    !zip -r embedder.zip embedder/
    files.download("embedder.zip")
    print("Download initiated for all model components.")
except Exception as e:
    print(f"Error during download: {e}")

Saving UpdatedResumeDataSet.csv to UpdatedResumeDataSet.csv
Generating TF-IDF embeddings...
Loading BERT model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

<IPython.core.display.Javascript object>

  adding: embedder/ (stored 0%)
  adding: embedder/model.safetensors (deflated 9%)
  adding: embedder/special_tokens_map.json (deflated 80%)
  adding: embedder/config.json (deflated 47%)
  adding: embedder/config_sentence_transformers.json (deflated 41%)
  adding: embedder/vocab.txt (deflated 53%)
  adding: embedder/tokenizer_config.json (deflated 73%)
  adding: embedder/2_Normalize/ (stored 0%)
  adding: embedder/modules.json (deflated 62%)
  adding: embedder/tokenizer.json (deflated 71%)
  adding: embedder/sentence_bert_config.json (deflated 9%)
  adding: embedder/1_Pooling/ (stored 0%)
  adding: embedder/1_Pooling/config.json (deflated 59%)
  adding: embedder/README.md (deflated 64%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated for all model components.
